# 03 — Inference Comparison: Base vs. LoRA Fine-Tuned

**Project:** Diffusion LoRA Fine-Tuning on Product Images  
**Author:** Adebanji Oluwatimileyin Adelowo

This notebook runs the same set of evaluation prompts through:
1. Base SDXL (no fine-tuning)
2. LoRA fine-tuned SDXL

Then assembles side-by-side comparison grids for the results section.

**Key constraint:** Use the same seed for both models so the only variable is the LoRA adapter.

In [ ]:
import torch
from pathlib import Path
from PIL import Image
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from diffusers import DiffusionPipeline

LORA_WEIGHTS = "../outputs/lora_weights/"
BASE_MODEL = "stabilityai/stable-diffusion-xl-base-1.0"
OUTPUT_DIR = Path("../results/inference/")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SEED = 42
NUM_IMAGES = 4      # images per prompt
STEPS = 30
CFG = 7.5

EVAL_PROMPTS = [
    "a product photo of sks eyewear on white background, studio lighting, sharp focus",
    "a product photo of sks eyewear, side profile, minimal background",
    "sks eyewear displayed on a clean surface, editorial photography",
    "close-up of sks eyewear temple detail, macro photography",
]

NEGATIVE = "blurry, low quality, distorted, cartoon, painting, illustration, bokeh, grainy"

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

In [ ]:
def generate(pipe, prompts, n, steps, cfg_scale, seed, negative):
    """Run inference for all prompts and return flat list of images."""
    generator = torch.Generator(device=device).manual_seed(seed)
    pipe.set_progress_bar_config(disable=True)
    all_images = []
    for prompt in prompts:
        result = pipe(
            prompt=prompt,
            negative_prompt=negative,
            num_images_per_prompt=n,
            num_inference_steps=steps,
            guidance_scale=cfg_scale,
            generator=generator,
        )
        all_images.extend(result.images)
    return all_images


def load_pipeline(model_id):
    pipe = DiffusionPipeline.from_pretrained(
        model_id,
        torch_dtype=torch.float16,
        variant="fp16",
        use_safetensors=True,
    ).to(device)
    return pipe

In [ ]:
## Generate: Base model
print("Loading base SDXL...")
base_pipe = load_pipeline(BASE_MODEL)
base_images = generate(base_pipe, EVAL_PROMPTS, NUM_IMAGES, STEPS, CFG, SEED, NEGATIVE)
del base_pipe
torch.cuda.empty_cache()
print(f"Base model: {len(base_images)} images generated")

In [ ]:
## Generate: LoRA fine-tuned model
print("Loading LoRA fine-tuned model...")
lora_pipe = load_pipeline(BASE_MODEL)
lora_pipe.load_lora_weights(LORA_WEIGHTS)
lora_images = generate(lora_pipe, EVAL_PROMPTS, NUM_IMAGES, STEPS, CFG, SEED, NEGATIVE)
del lora_pipe
torch.cuda.empty_cache()
print(f"LoRA model: {len(lora_images)} images generated")

In [ ]:
## Save individual images
for i, img in enumerate(base_images):
    img.save(OUTPUT_DIR / f"base_{i:03d}.png")
for i, img in enumerate(lora_images):
    img.save(OUTPUT_DIR / f"lora_{i:03d}.png")

print(f"Individual images saved to {OUTPUT_DIR}")

In [ ]:
## Build comparison grid
# Rows = prompts, Left half = base, Right half = LoRA

thumb = 256
n_prompts = len(EVAL_PROMPTS)
n_cols = NUM_IMAGES * 2

fig, axes = plt.subplots(n_prompts, n_cols, figsize=(n_cols * 2, n_prompts * 2.5))

for row_i, prompt in enumerate(EVAL_PROMPTS):
    for col_i in range(NUM_IMAGES):
        idx = row_i * NUM_IMAGES + col_i

        # Base column
        ax_base = axes[row_i, col_i]
        ax_base.imshow(base_images[idx])
        ax_base.axis("off")
        if row_i == 0:
            ax_base.set_title(f"BASE {col_i+1}", fontsize=8, color="#555")

        # LoRA column
        ax_lora = axes[row_i, NUM_IMAGES + col_i]
        ax_lora.imshow(lora_images[idx])
        ax_lora.axis("off")
        if row_i == 0:
            ax_lora.set_title(f"LoRA {col_i+1}", fontsize=8, color="#1a7fbf")

    # Prompt label on left
    axes[row_i, 0].set_ylabel(
        f"P{row_i+1}: {prompt[:30]}...", fontsize=7, rotation=0,
        labelpad=120, va="center"
    )

plt.suptitle("Base SDXL vs. LoRA Fine-Tuned — Product Image Comparison", fontsize=12, y=1.02)
plt.tight_layout()
grid_path = Path("../results/comparison_grid.png")
plt.savefig(grid_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"Comparison grid saved to {grid_path}")
print("\nNEXT: Run 04_results_analysis.ipynb")